In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
pip install google-genai

In [ ]:
import pandas as pd
import json
from google import genai
from google.genai import types
from tqdm import tqdm
import time
import os

# 1. Khởi tạo Client
client = genai.Client(api_key="...điền key vào đây")

# 2. Đọc dữ liệu từ file Excel
df = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/dt41-60.xlsx")

system_instruction = "Bạn là chuyên gia phân tích tài liệu y học học thuật."
output_file = "dataset_finetune_y_sinh_41-60.jsonl"

# KHÔNG xóa file nếu đang chạy dở, dùng append ("a") để nối tiếp

# 3. Lặp qua từng bài báo
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Đang sinh dữ liệu"):
    noi_dung = str(row['noi_dung_bai_bao'])

    if pd.isna(noi_dung) or len(noi_dung) < 100:
        continue

    text_input = noi_dung[:150000]

    user_prompt = f"""Nhiệm vụ:
Đọc nội dung tài liệu/bài báo cáo y học được cung cấp và tạo bản tóm tắt theo 3 cấp độ khác nhau.

YÊU CẦU CHUNG:
- Không bịa thêm dữ liệu ngoài tài liệu. Nếu tài liệu không có thông tin nào thì ghi: “Không đề cập”.
- Giữ nguyên các thuật ngữ y khoa quan trọng. Nếu có số liệu nghiên cứu, thuốc, chỉ số xét nghiệm → giữ nguyên đơn vị, không làm tròn tùy tiện.
- Nếu có guideline hoặc trial name → giữ nguyên tên tiếng Anh.
- Văn phong mạch lạc, logic. Ưu tiên nội dung lâm sàng, cơ chế bệnh sinh, chẩn đoán, điều trị và kết luận.

========================
LEVEL 1 — SƠ LƯỢC
Mục tiêu: Tạo bản tóm tắt ngắn gọn
- Chỉ dùng các gạch đầu dòng ngắn. Mỗi ý tối đa 1–2 dòng.
- Tập trung: Chủ đề chính, Cơ chế bệnh/chẩn đoán/điều trị, Kết luận quan trọng.
- Không diễn giải dài, Không phân tích sâu.

========================
LEVEL 2 — DỄ HIỂU
Mục tiêu: Giải thích tài liệu bằng ngôn ngữ đời thường để người không chuyên vẫn hiểu.
- Văn phong diễn giải súc tích rõ ràng, ngôn ngữ đời thường, không so sánh/ẩn dụ. Giải thích thuật ngữ ở mức độ đơn giản.
- Giải thích: Cơ chế bệnh sinh, Tác dụng thuốc, Ý nghĩa xét nghiệm, Tiến triển bệnh theo cách dễ hiểu.
- Không được làm sai bản chất y học hoặc bỏ mất ý quan trọng.
- Độ dài: 4–10 đoạn tùy độ phức tạp tài liệu.

========================
LEVEL 3 — CHUYÊN SÂU
Mục tiêu: Viết bản tóm tắt thành các đoạn văn học thuật hoàn chỉnh cho người có chuyên môn y khoa.
- Giữ nguyên: Thuật ngữ Latin, Tên thuốc, Chỉ số xét nghiệm, Tên guideline/trial.
- Phân tích logic theo cấu trúc: Tổng quan -> Cơ chế bệnh sinh -> Triệu chứng/lâm sàng -> Chẩn đoán -> Điều trị -> Kết quả nghiên cứu -> Kết luận chuyên môn.
- Văn phong học thuật, chặt chẽ. Không đơn giản hóa thuật ngữ chuyên môn.

========================
TRẢ VỀ DUY NHẤT ĐỊNH DẠNG JSON SAU (Không kèm giải thích gì thêm):
{{
  "chu_de": "<Trích xuất Chủ đề chính của tài liệu>",
  "so_luoc": "<Nội dung LEVEL 1>",
  "de_hieu": "<Nội dung LEVEL 2>",
  "chuyen_sau": "<Nội dung LEVEL 3>"
}}

========================
TÀI LIỆU GỐC:
{text_input}"""

    max_retries = 4 # Cho phép thử lại 4 lần

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=0.2,
                    response_mime_type="application/json",
                )
            )

            # Nếu JSON bị lỗi ngoặc kép/kí tự đặc biệt, lệnh loads sẽ báo lỗi và nhảy xuống except
            result = json.loads(response.text)

            levels = {
                "Sơ lược": result.get("so_luoc", ""),
                "Dễ hiểu": result.get("de_hieu", ""),
                "Chuyên sâu": result.get("chuyen_sau", "")
            }

            # LƯU FILE
            with open(output_file, "a", encoding="utf-8") as f:
                for level_name, summary in levels.items():
                    if summary:
                        chat_ml_format = {
                            "messages": [
                                {"role": "system", "content": "Bạn là trợ lý AI chuyên tóm tắt tài liệu học thuật y sinh tiếng Việt."},
                                {"role": "user", "content": f"Hãy tóm tắt văn bản sau theo mức độ '{level_name}':\n\n{text_input}"},
                                {"role": "assistant", "content": summary}
                            ]
                        }
                        f.write(json.dumps(chat_ml_format, ensure_ascii=False) + "\n")

            # Thành công thì đợi 10s rồi qua bài mới (để tránh đầy Token Per Minute)
            time.sleep(10)
            break

        except json.JSONDecodeError:
            print(f"\n[Lỗi JSON] Mô hình sinh sai cấu trúc ở dòng {index}. Đang thử sinh lại... (Lần {attempt + 1}/{max_retries})")
            time.sleep(5)

        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                # Đợi hẳn 60s để Google xả sạch giới hạn TPM (Token Per Minute)
                print(f"\n[Quá tải API] Đợi 60 giây để khôi phục... (Lần {attempt + 1}/{max_retries})")
                time.sleep(60)
            else:
                print(f"\n[Lỗi] Bài báo dòng {index} gặp lỗi: {e}")
                time.sleep(5)

print(f"\nHoàn tất! Dữ liệu được lưu tại {output_file}")

In [ ]:
# 1. Đọc file JSONL
data_list = []
with open("dataset_finetune_y_sinh_41-60.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)ễ
        # Bóc tách nội dung từ cấu trúc ChatML
        user_prompt = item["messages"][1]["content"]
        assistant_response = item["messages"][2]["content"]

        # Xác định level tóm tắt từ câu lệnh
        level = "Sơ lược" if "Sơ lược" in user_prompt else ("Dễ hiểu" if "Dễ hiểu" in user_prompt else "Chuyên sâu")

        data_list.append({
            "Mức độ": level,
            "Bản tóm tắt": assistant_response
        })

# 2. Xuất ra file Excel để review bằng mắt
df_review = pd.DataFrame(data_list)
df_review.to_excel("Kiem_tra_tom_tat.xlsx", index=False)
print("Đã xuất file Kiem_tra_tom_tat.xlsx. Hãy mở bằng Excel để xem nhé!")

Đã xuất file Kiem_tra_tom_tat_v4.xlsx. Hãy mở bằng Excel để xem nhé!
